In [1]:
import pdfplumber
import re

pdf_path = "bns.pdf"
raw_text = ""
BATCH_SIZE = 20
all_sections = []

with pdfplumber.open(pdf_path) as pdf:
    for start in range(15, len(pdf.pages), BATCH_SIZE):
        
        raw_text = ""
        batch_start_page = start + 1  

        for i, page in enumerate(pdf.pages[start:start+BATCH_SIZE]):
            text = page.extract_text()
            if text:
                raw_text += f"##PAGE_BREAK_{start + i + 1}##\n" + text + "\n"  

        
        clean_text = raw_text
        clean_text = re.sub(r'THE BHARATIYA NYAYA SANHITA, 2023\s*\n'
            r'ACT NO\. 45 OF 2023\s*\n'
            r'\[.*?\]\s*\n'
            r'An Act.*?thereto\.\s*\n'
            r'BE it enacted.*?follows:––\s*\n',
            '', clean_text, flags=re.DOTALL)
        clean_text = re.sub(r'\n\d+\.\s+\d+(?:st|nd|rd|th) day of [A-Za-z]+.*?sec\.\s*\d+\([ivxIVX]+\)\.', '', clean_text, flags=re.DOTALL)
        clean_text = re.sub(r'(?<=[a-z])\d(?=\s)', '', clean_text)
        clean_text = re.sub(r'^\d{1,3}\s*$', '', clean_text, flags=re.MULTILINE)
        clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
        clean_text = clean_text.strip()
        clean_text = re.sub(r'Bharatiya Nyaya Sanhita.*?\n', '', clean_text)
        clean_text = re.sub(r'\n\s*\d+\s*\n', '\n', clean_text)

        sections = re.split(r'\n(?=\d+\.\s)', clean_text)
        current_chapter = "Unknown"
        current_page = batch_start_page  

        for sec in sections:
            
            page_marker = re.search(r'##PAGE_BREAK_(\d+)##', sec)
            if page_marker:
                current_page = int(page_marker.group(1))
            sec_clean = re.sub(r'##PAGE_BREAK_\d+##\n?', '', sec)  

            chapter_match = re.search(r'CHAPTER\s+[IVX]+.*', sec_clean)
            if chapter_match:
                current_chapter = chapter_match.group(0)

            match = re.match(r'(\d+)\.\s*(.*)', sec_clean, re.DOTALL)
            if match:
                all_sections.append({
                    "chapter": current_chapter,
                    "section_number": match.group(1),
                    "section_text": match.group(2).strip(),
                    "page_number": current_page  
                })

for s in all_sections[:]:
    print(s["section_number"], "|", s["chapter"], "|", s["page_number"])

1 | CHAPTER I | 16
2 | CHAPTER I | 17
3 | CHAPTER II | 21
4 | CHAPTER II | 22
5 | CHAPTER II | 22
6 | CHAPTER II | 22
7 | CHAPTER II | 22
8 | CHAPTER II | 23
9 | CHAPTER II | 23
10 | CHAPTER II | 23
11 | CHAPTER II | 24
12 | CHAPTER II | 24
13 | CHAPTER III | 24
14 | CHAPTER III | 24
15 | CHAPTER III | 24
16 | CHAPTER III | 24
17 | CHAPTER III | 25
18 | CHAPTER III | 25
19 | CHAPTER III | 25
20 | CHAPTER III | 25
21 | CHAPTER III | 25
22 | CHAPTER III | 25
23 | CHAPTER III | 25
24 | CHAPTER III | 26
25 | CHAPTER III | 26
26 | CHAPTER III | 26
27 | CHAPTER III | 26
28 | CHAPTER III | 27
29 | CHAPTER III | 27
30 | CHAPTER III | 27
31 | CHAPTER III | 28
32 | CHAPTER III | 28
33 | CHAPTER III | 28
34 | CHAPTER III | 28
35 | CHAPTER III | 28
36 | CHAPTER III | 28
37 | CHAPTER III | 29
38 | CHAPTER III | 29
39 | CHAPTER III | 29
40 | CHAPTER III | 29
41 | CHAPTER III | 30
42 | CHAPTER III | 30
43 | CHAPTER III | 30
44 | CHAPTER IV | 30
45 | CHAPTER IV | 31
46 | CHAPTER IV | 32
47 | CHAPTER I

In [2]:
print("Total sections:", len(all_sections))

Total sections: 363


In [3]:
import sqlite3

conn = sqlite3.connect("legal_agent.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS bns_sections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    chapter TEXT,
    section_number TEXT,
    section_text TEXT
)
""")
conn.commit()

In [4]:
import os
[f for f in os.listdir(".") if f.endswith(".db")]

['legal_agent.db']

In [5]:
for s in all_sections:
    cursor.execute("""
    INSERT INTO bns_sections (chapter, section_number, section_text)
    VALUES (?, ?, ?)
    """, ("Chapter 1", s["section_number"], s["section_text"]))

conn.commit()

In [6]:
def get_bns_section(section_number):
    conn = sqlite3.connect("legal_agent.db")
    cursor = conn.cursor()
    cursor.execute("""
    SELECT section_text, page_number FROM bns_sections
    WHERE section_number = ?
    """, (section_number,))
    result = cursor.fetchone()
    conn.close()
    return result

print(get_bns_section("4"))

('Punishments. —The punishments to which offenders are liable under the provisions of this\nSanhita are—\n(a) Death;\n(b) Imprisonment for life;\n\n(c) Imprisonment, which is of two descriptions, namely:—\n(1) Rigorous, that is, with hard labour;\n(2) Simple;\n(d) Forfeiture of property;\n(e) Fine;\n(f) Community Service.', 3)


## Building a legal query interpreter

In [11]:
import re

def extract_section_from_query(query):
    match = re.search(r'section\s+(\d+)', query, re.IGNORECASE)
    if match:
        return match.group(1)
    return None

print(extract_section_from_query("What is section 2 of BNS?"))

2


In [12]:
def answer_query(query):
    section = extract_section_from_query(query)
    
    if section:
        result = get_bns_section(section)
        if result:
            return f"Section {section}:\n\n{result[0]}"
        else:
            return "Section not found."
    
    return "Query type not supported yet."

print(answer_query("What is section 1?"))

Section 1:

Short title, commencement and application.––(1) This Act may be called the Bharatiya Nyaya
Sanhita, 2023.
(2) It shall come into force on such date as the Central Government may, by notification in the
Official Gazette, appoint, and different dates may be appointed for different provisions of this Sanhita.
(3) Every person shall be liable to punishment under this Sanhita and not otherwise for every act or
omission contrary to the provisions thereof, of which he shall be guilty within India.
(4) Any person liable, by any law for the time being in force in India, to be tried for an offence
committed beyond India shall be dealt with according to the provisions of this Sanhita for any act
committed beyond India in the same manner as if such act had been committed within India.
(5) The provisions of this Sanhita shall also apply to any offence committed by—
(a) any citizen of India in any place without and beyond India;
(b) any person on any ship or aircraft registered in India 

## smart retrieval

In [13]:
def keyword_search(query):
    cursor.execute("""
    SELECT section_number, section_text
    FROM bns_sections
    WHERE section_text LIKE ?
    """, (f"%{query}%",))
    
    return cursor.fetchall()

def answer_query(query):
    section = extract_section_from_query(query)
    
    if section:
        result = get_bns_section(section)
        if result:
            return f"Section {section}:\n\n{result[0]}"
    
    results = keyword_search(query)
    
    if results:
        response = "Relevant Sections:\n\n"
        for sec in results[:3]:
            response += f"Section {sec[0]}:\n{sec[1][:200]}...\n\n"
        return response
    
    return "No relevant legal section found."

## semantic search (phase 4)

In [14]:
LEGAL_KEYWORDS = {
    "stealing": "theft dishonestly movable property punishment section",
    "theft": "theft dishonestly movable property punishment",
    "fraud": "fraudulently deception cheating punishment",
    "murder": "murder causing death punishment",
    "assault": "assault injury offence punishment",
    "cheating": "cheating deception fraud punishment",
    "police":   "arrest offence",
}

In [15]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

section_texts = [s["section_text"] for s in all_sections]
embeddings = model.encode(section_texts)

C:\Users\PrajjnaaRayChoudhury\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3446.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
#semantic search
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query):
    query_embedding = model.encode([query])
    
    scores = cosine_similarity(embeddings, query_embedding).flatten()
    
    top_indices = scores.argsort()[-3:][::-1]
    
    results = []
    for idx in top_indices:
        results.append(all_sections[idx])
        
    return results, scores

In [17]:
from transformers import pipeline

generator = pipeline("text-generation", model="sshleifer/tiny-gpt2", max_length=None)
def rewrite_query(query):
    """Expand query using keyword map — no LLM needed."""
    q = query.lower()
    for key, expansion in LEGAL_KEYWORDS.items():
        if key in q:
            return expansion
    return query

    
def explain_simple(text):
    """Extract the first 1-2 clean sentences as a plain-English summary."""
    text = re.sub(r'—+', '-', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    sentences = re.split(r'(?<=[.;])\s+', text)
    
    summary = []
    for s in sentences:
        s = s.strip()
        if len(s) > 30:
            summary.append(s)
        if len(summary) == 2:
            break
    
    return ' '.join(summary) if summary else text[:200]

Loading weights: 100%|███████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 9304.28it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [32]:
def semantic_agent(query):
    
    q = query.lower()
    
    for key in LEGAL_KEYWORDS:
        if key in q:
            query = LEGAL_KEYWORDS[key]
            break
    
    improved_query = rewrite_query(query)
    
    results, scores = semantic_search(improved_query)
    
    return format_response(results)

In [33]:
def format_response(results):
    response = "💡 In simple terms:\n"
    response += explain_simple(results[0]["section_text"]) + "\n\n"
    
    response += "⚖️ Legal Insight:\n\n"
    
    for r in results:
        section_num = r['section_number']
        page_num = r.get('page_number', 1)
        link = f"/static/bns.pdf#page={page_num}"
        response += f"📌 <a href='{link}' target='_blank'>Section {section_num}</a>:\n"
        response += r['section_text'][:] + "...\n\n"
    
    return response

## agent layer

In [34]:
#query agent
def classify_query(query):
    q = query.lower()

    if "section" in q:
        return "section_lookup"
    
    elif "define" in q or "meaning" in q:
        return "definition"
    
    elif len(q.split()) <= 2:
        return "definition"
    
    else:
        return "semantic"

## Manual query check

In [35]:
query = "What is punishment under BNS?"

results, scores = semantic_search(query)

score = max(scores)

threshold = 0.3

if score < threshold:
    print("Not confident enough")
else:
    print(results)

[{'chapter': 'CHAPTER II', 'section_number': '4', 'section_text': 'Punishments. —The punishments to which offenders are liable under the provisions of this\nSanhita are—\n(a) Death;\n(b) Imprisonment for life;\n\n(c) Imprisonment, which is of two descriptions, namely:—\n(1) Rigorous, that is, with hard labour;\n(2) Simple;\n(d) Forfeiture of property;\n(e) Fine;\n(f) Community Service.', 'page_number': 22}, {'chapter': 'CHAPTER XII', 'section_number': '200', 'section_text': 'Punishment for non-treatment of victim.—Whoever, being in charge of a hospital, public or\nprivate, whether run by the Central Government, the State Government, local bodies or any other person,\ncontravenes the provisions of section 397 of the Bharatiya Nagarik Suraksha Sanhita, 2023, shall be\npunished with imprisonment for a term which may extend to one year, or with fine, or with both.', 'page_number': 69}, {'chapter': 'CHAPTER IV', 'section_number': '49', 'section_text': 'Punishment of abetment if act abetted 

## Legal agent building

In [36]:
def split_definitions(section_text):
    parts = re.split(r'\(\d+\)', section_text)
    return [p.strip() for p in parts if p.strip()]

def filter_definition(section_text, keyword):
    parts = split_definitions(section_text)
    
    results = []
    
    for part in parts:
        if keyword.lower() in part.lower():
            results.append(part)
    
    return results

def extract_keyword(query):
    words = query.lower().split()
    
    stopwords = ["define", "what", "is", "the", "of"]
    
    keywords = [w for w in words if w not in stopwords]
    
    return keywords[0] if keywords else query

In [37]:
def definition_agent(query):
    
    words = query.lower().split()
    
    if len(words) == 1:
        keyword = words[0]   # ✅ direct keyword
    else:
        keyword = extract_keyword(query)
    
    result = get_bns_section("2")
    
    if result:
        filtered = filter_definition(result[0], keyword)
        
        if filtered:
            response = "📖 Definition:\n\n"
            for f in filtered:
                response += f + "\n\n"
            return response
    
    return "No relevant definition found."

In [38]:
def add_safety(response):
    return response + "\n\n⚠️ This is general legal information. Consult a lawyer if needed."

In [39]:
def advice_layer(query):

    q = query.lower()

    if "caught" in q or "police" in q:
        return "\t\t🚨 What you should do:\n- Stay calm\n- Do not panic\n- Do not immediately admit guilt\t- Ask for legal help\t"

    return ""

In [40]:
def legal_agent(query):
    
    intent = classify_query(query)
    
    if intent == "definition":
        response = definition_agent(query)
    
    elif intent == "section_lookup":
        response = answer_query(query)

    elif intent == "legal_help":   
        response = semantic_agent(query)
    
    else:
        response = semantic_agent(query)
    response += advice_layer(query)
    
    return add_safety(response)   

In [47]:
print(legal_agent("define fraud"))

📖 Definition:

“fraudulently” means doing anything with the intention to defraud but not otherwise;



⚠️ This is general legal information. Consult a lawyer if needed.


In [48]:
print(legal_agent("dishonest"))

📖 Definition:

“dishonestly” means doing anything with the intention of causing wrongful gain to one
person or wrongful loss to another person;



⚠️ This is general legal information. Consult a lawyer if needed.


In [49]:
legal_agent("I was caught stealing")

"💡 In simple terms:\nDishonest or fraudulent removal or concealment of property.-Whoever dishonestly or fraudulently conceals or removes any property of himself or any other person, or dishonestly or fraudulently assists in the concealment or removal thereof, or dishonestly releases any demand or claim to which he is entitled, shall be punished with imprisonment of either description for a term which may extend to three years, or with fine, or with both.\n\n⚖️ Legal Insight:\n\n📌 <a href='/static/bns.pdf#page=97' target='_blank'>Section 323</a>:\nDishonest or fraudulent removal or concealment of property.—Whoever dishonestly or\nfraudulently conceals or removes any property of himself or any other person, or dishonestly or\nfraudulently assists in the concealment or removal thereof, or dishonestly releases any demand or claim\nto which he is entitled, shall be punished with imprisonment of either description for a term which may\nextend to three years, or with fine, or with both.\nOf m

# Flask API

In [50]:
from flask import Flask, request, jsonify, render_template, send_from_directory

app = Flask(__name__)

@app.route('/static/bns.pdf')
def static_files():
    return send_from_directory('static', 'bns.pdf')

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/legal", methods=["POST"])
def legal_api():
    query = request.json.get("query")
    result = legal_agent(query)
    return jsonify({"response": result})

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Apr/2026 10:15:53] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:16:24] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:16:45] "GET /static/bns.pdf HTTP/1.1" 304 -
127.0.0.1 - - [02/Apr/2026 10:17:13] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:17:42] "GET /static/bns.pdf HTTP/1.1" 304 -
127.0.0.1 - - [02/Apr/2026 10:18:20] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:18:57] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:19:08] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:20:10] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:20:44] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:21:14] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:21:52] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:23:07] "POST /legal HTTP/1.1" 200 -
127.0.0.1 - - [02/Apr/2026 10:24:15] "POST /legal HTTP/1.1" 200 -
127.0.0.1

#### static bns file

In [115]:
import fitz
import re

def build_section_page_map(pdf_path):
    section_page_map = {}
    doc = fitz.open(pdf_path)
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        if text:
            matches = re.findall(r'(?:^|\n)\s*(\d+)\.\s+\w', text)
            for section_num in matches:
                if section_num not in section_page_map:
                    section_page_map[section_num] = page_num
    doc.close()
    return section_page_map

page_map = build_section_page_map("bns.pdf")
print(page_map)

{'1': 3, '2': 3, '3': 3, '4': 3, '5': 3, '6': 3, '7': 3, '8': 3, '9': 3, '10': 3, '11': 3, '12': 3, '13': 3, '14': 3, '15': 3, '16': 3, '17': 3, '18': 3, '19': 3, '20': 3, '21': 3, '22': 3, '23': 3, '24': 3, '25': 4, '26': 4, '27': 4, '28': 4, '29': 4, '30': 4, '31': 4, '32': 4, '33': 4, '34': 4, '35': 4, '36': 4, '37': 4, '38': 4, '39': 4, '40': 4, '41': 4, '42': 4, '43': 4, '44': 4, '45': 4, '46': 4, '47': 4, '48': 4, '49': 4, '50': 4, '51': 4, '52': 4, '53': 4, '54': 4, '55': 5, '56': 5, '57': 5, '58': 5, '59': 5, '60': 5, '61': 5, '62': 5, '63': 5, '64': 5, '65': 5, '66': 5, '67': 5, '68': 5, '69': 5, '70': 5, '71': 5, '72': 5, '73': 5, '74': 5, '75': 5, '76': 5, '77': 5, '78': 5, '79': 5, '80': 5, '81': 6, '82': 6, '83': 6, '84': 6, '85': 6, '86': 6, '87': 6, '88': 6, '89': 6, '90': 6, '91': 6, '92': 6, '93': 6, '94': 6, '95': 6, '96': 6, '97': 6, '98': 6, '99': 6, '100': 6, '101': 6, '102': 6, '103': 6, '104': 6, '105': 6, '106': 6, '107': 6, '108': 6, '109': 6, '110': 6, '111': 

In [116]:
conn = sqlite3.connect("legal_agent.db")
cursor = conn.cursor()

try:
    cursor.execute("ALTER TABLE bns_sections ADD COLUMN page_number INTEGER")
    conn.commit()
except:
    pass  # column already exists, ignore

for section_num, page_num in page_map.items():
    cursor.execute("UPDATE bns_sections SET page_number = ? WHERE section_number = ?",
                   (page_num, section_num))
conn.commit()
conn.close()
print("Done")

Done


In [94]:
import os
print(os.getcwd())

C:\Users\PrajjnaaRayChoudhury\Desktop\clg stuff\sem 6\ai_PROJECT


In [ ]:
import os
print(os.path.exists("static/bns.pdf"))  # should print True
